In [12]:
import pandas as pd
df = pd.read_csv("sold_flagged_full.csv")
print(df.shape)

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_9631/3471475793.py:2: DtypeWarning: Columns (0: FireplaceYN, 1: OriginatingSystemName, 2: OriginatingSystemSubName, 3: latfilled, 4: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("sold_flagged_full.csv")


(414184, 77)


In [13]:
flag_cols = [
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag",
    "invalid_close_price_flag",
    "invalid_living_area_flag",
    "invalid_days_on_market_flag",
    "invalid_bedrooms_flag",
    "invalid_bathrooms_flag",
    "missing_geo_flag",
    "zero_geo_flag",
    "invalid_longitude_flag",
    "out_of_ca_flag"
]

df[flag_cols].sum().sort_values(ascending=False)

missing_geo_flag               15956
negative_timeline_flag           270
purchase_after_close_flag        240
invalid_living_area_flag         150
out_of_ca_flag                    88
listing_after_close_flag          61
invalid_days_on_market_flag       47
invalid_longitude_flag            31
zero_geo_flag                     27
invalid_close_price_flag           1
invalid_bedrooms_flag              0
invalid_bathrooms_flag             0
dtype: int64

In [14]:
 # Convert numeric columns
numeric_cols = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print("\nNumeric conversion complete")
print(df[numeric_cols].dtypes)


Numeric conversion complete
ClosePrice      float64
LivingArea      float64
DaysOnMarket      int64
dtype: object


In [ ]:
# These flags were already created in previous weeks

rule_flags = [
    "invalid_close_price_flag",
    "invalid_living_area_flag",
    "invalid_days_on_market_flag",
    "invalid_numeric_flag"
]

for col in rule_flags:

    if col in df.columns:

        # Convert to Boolean
        df[col] = (
            df[col]
            .astype(str)
            .str.lower()
            .isin(["true", "1", "yes"])
        )
        print(f"{col} loaded successfully")

    else:
        print(f"Warning: {col} not found")


invalid_close_price_flag loaded successfully
invalid_living_area_flag loaded successfully
invalid_days_on_market_flag loaded successfully
invalid_numeric_flag loaded successfully


In [16]:
#  Define IQR outlier function

def add_iqr_flag(df, column_name):

    # Calculate quartiles
    Q1 = df[column_name].quantile(0.25)
    Q3 = df[column_name].quantile(0.75)

   
    IQR = Q3 - Q1

  
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

   
    flag_col = f"{column_name}_iqr_outlier_flag"
 # Flag outliers
    df[flag_col] = (
        (df[column_name] < lower) |
        (df[column_name] > upper)
    )

    # Print IQR summary
    print("\n===================================")
    print(f"{column_name} IQR Summary")
    print("===================================")

    print(f"Q1: {Q1}")
    print(f"Q3: {Q3}")
    print(f"IQR: {IQR}")

    print(f"Lower Bound: {lower}")
    print(f"Upper Bound: {upper}")

    print(
        f"Outliers Flagged: {df[flag_col].sum()}"
    )

    return df

In [17]:
#Apply IQR outlier detection
# Apply IQR filtering to:
# ClosePrice
# LivingArea
# DaysOnMarket

df = add_iqr_flag(df, "ClosePrice")

df = add_iqr_flag(df, "LivingArea")

df = add_iqr_flag(df, "DaysOnMarket")


ClosePrice IQR Summary
Q1: 575000.0
Q3: 1300000.0
IQR: 725000.0
Lower Bound: -512500.0
Upper Bound: 2387500.0
Outliers Flagged: 30782

LivingArea IQR Summary
Q1: 1248.0
Q3: 2219.0
IQR: 971.0
Lower Bound: -208.5
Upper Bound: 3675.5
Outliers Flagged: 18218

DaysOnMarket IQR Summary
Q1: 8.0
Q3: 48.0
IQR: 40.0
Lower Bound: -52.0
Upper Bound: 108.0
Outliers Flagged: 31826


In [18]:
# Create combined IQR outlier flag
# If any numeric field is an outlier,
# mark the row as True

df["any_iqr_outlier_flag"] = (

    df["ClosePrice_iqr_outlier_flag"] |

    df["LivingArea_iqr_outlier_flag"] |

    df["DaysOnMarket_iqr_outlier_flag"]

)

print("\nCombined IQR flag created")

print(
    df["any_iqr_outlier_flag"]
    .value_counts()
)



Combined IQR flag created
any_iqr_outlier_flag
False    349099
True      65085
Name: count, dtype: int64


In [19]:
# Create final exclusion flag- Combine:
# rule invalid records + IQR outliers
if "invalid_numeric_flag" in df.columns:

    df["exclude_from_analysis_flag"] = (

        df["invalid_numeric_flag"] |

        df["any_iqr_outlier_flag"]

    )

else:

    df["exclude_from_analysis_flag"] = (
        df["any_iqr_outlier_flag"]
    )

print("\nFinal exclusion flag created")

print(
    df["exclude_from_analysis_flag"]
    .value_counts()
)


Final exclusion flag created
exclude_from_analysis_flag
False    348979
True      65205
Name: count, dtype: int64


In [20]:
#  Create clean filtered dataset
# Keep only records
# that are NOT excluded

df_filtered = df[
    ~df["exclude_from_analysis_flag"]
].copy()

print("\nFiltered dataset created")

print("Original rows:", len(df))

print("Filtered rows:", len(df_filtered))


Filtered dataset created
Original rows: 414184
Filtered rows: 348979


In [21]:
# Compare before and after filtering

comparison_rows = []

fields = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]

for col in fields:

    outlier_flag_col = f"{col}_iqr_outlier_flag"

    comparison_rows.append({

        "Field": col,

        "Median Before": df[col].median(),

        "Median After": df_filtered[col].median(),

        "Mean Before": df[col].mean(),

        "Mean After": df_filtered[col].mean(),

      

    })

comparison = pd.DataFrame(comparison_rows)
print("Before vs After Filtering Comparison")

print(comparison)

Before vs After Filtering Comparison
          Field  Median Before  Median After   Mean Before     Mean After
0    ClosePrice       823000.0      785000.0  1.190837e+06  898806.684445
1    LivingArea         1642.0        1569.0  1.903791e+03    1675.064201
2  DaysOnMarket           18.0          16.0  3.736125e+01      26.271054
